### KNN Imputer

**KNN Imputer** fills a missing value by looking at the *K nearest rows* (neighbours) that have an observed value for that feature. It uses the other available features to find rows that are similar.

Unlike mean or median imputation, KNN imputation uses relationships between observations. In this notebook, it is applied to the numerical Titanic features `Age`, `Pclass`, and `Fare`.

#### How KNN Imputation works

1. For a row with a missing value, the imputer finds other rows that are most similar based on the features available for distance calculation.
2. It selects up to `n_neighbors` nearest rows that have a value for the feature being imputed.
3. It calculates a replacement from those neighbours' observed values.
4. The process is repeated for the missing values in the dataset.

The default distance is based on **NaN-aware Euclidean distance**, which can calculate distances using features observed in both rows.

**Advantages**
- Uses relationships between features and similar observations.
- Can preserve local patterns better than a single global mean or median.
- Supports different neighbour weighting strategies.

**Disadvantages**
- Can be computationally expensive on large datasets.
- Sensitive to feature scales and the choice of distance metric.
- Results depend on the number and quality of neighbours.
- Can perform poorly when rows are not meaningfully similar or many features are missing.

#### Important KNNImputer parameters

- **`n_neighbors`**: number of neighbours considered. This notebook uses `3`. A small value is more local but can be sensitive to noise; a larger value can smooth the imputed values.
- **`weights`**: controls each neighbour's contribution.
  - `'uniform'`: all selected neighbours contribute equally.
  - `'distance'`: closer neighbours contribute more than farther neighbours. This notebook uses `'distance'`.
- **`metric`**: distance measure used to find neighbours. The default is `nan_euclidean`.
- **`add_indicator`**: when set to `True`, appends missingness-indicator features to the transformed output.

#### Why feature scaling matters

KNN is distance-based. Features with larger numerical ranges can dominate distance calculations. For example, `Fare` may have a much wider range than `Pclass`.

Consider scaling numerical features before KNN imputation so that distance calculations are not dominated by one feature's scale. Scaling must be fitted using training data only. A preprocessing pipeline is a good way to keep the steps in the correct order and avoid data leakage.  

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.impute import KNNImputer,SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score

In [2]:
df = pd.read_csv('Datasets/titanic.csv')[['Age','Pclass','Fare','Survived']]

In [3]:
df.head()

,Age,Pclass,Fare,Survived
0,22.0,3,7.2500,0
1,38.0,1,71.2833,1
2,26.0,3,7.9250,1
3,35.0,1,53.1000,1
4,35.0,3,8.0500,0


In [4]:
df.isnull().mean() * 100

Age         19.86532
Pclass       0.00000
Fare         0.00000
Survived     0.00000
dtype: float64

In [5]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [6]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)

In [7]:
X_train.head()

,Age,Pclass,Fare
30,40.0,1,27.7208
10,4.0,3,16.7000
873,47.0,3,9.0000
182,9.0,3,31.3875
876,20.0,3,9.8458


#### Comparing KNN Imputation with Simple Imputation

The notebook compares :
- **KNN Imputer :** estimates missing values from nearby observations.
- **Simple Imputer :** uses the column mean by default (`strategy='mean'`).

Both approaches are evaluated with Logistic Regression using accuracy on the same held-out test split.

A higher test accuracy in one run does not prove that the method is always better. For a more reliable comparison, use cross-validation and consider additional metrics appropriate to the task.

In [8]:
knn = KNNImputer(n_neighbors=3,weights='distance')

X_train_trf = knn.fit_transform(X_train)
X_test_trf = knn.transform(X_test)

In [9]:
lr = LogisticRegression()

lr.fit(X_train_trf,y_train)

y_pred = lr.predict(X_test_trf)

accuracy_score(y_test,y_pred)

0.7039106145251397

In [10]:
# Comparision with Simple Imputer --> mean

si = SimpleImputer()

X_train_trf2 = si.fit_transform(X_train)
X_test_trf2 = si.transform(X_test)

In [11]:
lr = LogisticRegression()

lr.fit(X_train_trf2,y_train)

y_pred2 = lr.predict(X_test_trf2)

accuracy_score(y_test,y_pred2)

0.6927374301675978